# Cabouy - consolidation des chroniques

Trois sondes : **CTD** (Diver autonome : niveau, conductivité, température),
**TROLL** (Aqua TROLL : conductivité, température, turbidité, O2, chlorophylle,
**pas de niveau**), **OTT** (la sonde CTD de la centrale : niveau, conductivité,
température).

La centrale rapatrie aussi les voies du TROLL (`C2`, `T2`, `Turbi`, `O2`,
`Chlorophyl`) : c'est le **même capteur** que les exports VuSitu, un second chemin
d'acquisition, pas une quatrième sonde. Les deux sont réunis en cellule 8, sans
recalage.

Pour chaque grandeur, une liste `PERIODES_*` dit **quelle sonde est prioritaire sur
quelle période**. Elle est écrite en dur, juste au-dessus du graphe de correction.
La cellule 9 la propose une fois, à partir de la disponibilité réelle des sondes.

## 1. Imports

In [ ]:
import os
import re
import unicodedata
from io import StringIO

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go

## 2. Chemins d'accès

In [ ]:
BASE        = r"Y:\MISSIONS\Eau\1 - Projet Hydrogéologique Ouysse\3 - Hydrodynamique\0 - Stations en continu\Cabouy\Gaetan"
CTD_PATH    = os.path.join(BASE, r"Données brutes\CTD")
VUSITU_PATH = os.path.join(BASE, r"Données brutes\TROLL")
OTT_PATH    = os.path.join(BASE, r"Données brutes\OTT")
BARO_PATH   = r"Y:\MISSIONS\Eau\1 - Projet Hydrogéologique Ouysse\3 - Hydrodynamique\0 - Stations en continu\1 - Données BARO\Gourdon baro\Patm Calès et Thémines.xlsx"
PLUIE_PATH  = r"Y:\MISSIONS\Eau\1 - Projet Hydrogéologique Ouysse\3 - Hydrodynamique\0 - Stations en continu\Saint Sauveur\Gaetan\Données brutes\Pluie_BV_Ouysse.csv"

OLDDATA_PATH     = os.path.join(BASE, "Cabouy_consolide_OLD.xlsx")
UTC_CTD_PATH     = os.path.join(BASE, "UTC_CTD.xlsx")
UTC_TROLL_PATH   = os.path.join(BASE, "UTC_Troll.xlsx")
PUNCTUAL_NIVEAU  = os.path.join(BASE, "punctual_measurements.xlsx")
PUNCTUAL_CONDUCT = os.path.join(BASE, "punctual_measurements_conducti.xlsx")
SORTIE_CONSOLIDE = os.path.join(BASE, "Cabouy_consolide.xlsx")
SORTIE_FINALE    = os.path.join(BASE, "Cabouy_final.xlsx")
SORTIE_SVG       = os.path.join(BASE, "Graphes.svg")

PREFIXE_CTD = "Cabouy"
BARO_COL    = "Patm Ouysse Calès [hPa]"
PAS         = "1h"

## 3. Fonctions de lecture

Les pièges de format : en-tête Diver à une ligne variable et pied `END OF DATA`,
guillemets et numéros de série VuSitu, sentinelle `-99999` de la centrale, virgules
décimales, encodages. Un fichier absent de la table UTC est ignoré, avec son nom :
c'est la table qui se corrige.

In [ ]:
def _sans_accents(t):
    d = unicodedata.normalize("NFKD", str(t))
    return "".join(c for c in d if not unicodedata.combining(c)).lower()


def _lire_lignes(chemin):
    for enc in ("utf-8-sig", "utf-8", "cp1252", "latin1"):
        try:
            with open(chemin, "r", encoding=enc) as f:
                return f.read().splitlines(), enc
        except UnicodeDecodeError:
            continue
    raise ValueError(f"Aucun encodage ne convient pour {chemin}")


def _en_datetime(serie):
    """Garde le format qui convertit le plus de lignes."""
    txt = serie.astype("string").str.strip()
    meilleur, n_ok = None, -1
    for fmt in ("%Y/%m/%d %H:%M:%S", "%Y-%m-%d %H:%M:%S", "%d/%m/%Y %H:%M:%S", "%d/%m/%Y %H:%M"):
        e = pd.to_datetime(txt, format=fmt, errors="coerce")
        if e.notna().sum() > n_ok:
            meilleur, n_ok = e, e.notna().sum()
    if n_ok < len(txt):                      # format inconnu : lecture libre
        libre = pd.to_datetime(txt, errors="coerce", dayfirst=True)
        meilleur = libre if libre.notna().sum() > n_ok else meilleur
    return meilleur


def fichiers(path, motif):
    """Fichiers du dossier contenant `motif`, triés par numéro."""
    noms = [f for f in os.listdir(path)
            if motif.lower() in f.lower() and f.lower().endswith((".csv", ".txt", ".mon"))]
    return sorted(noms, key=lambda n: (int(re.findall(r"\d+", n)[0]) if re.findall(r"\d+", n) else 10 ** 9, n))


def lire_CTD(nom, path=CTD_PATH):
    """Export Diver : en-tête cherchée par contenu, pied END OF DATA reconnu,
    conductivité convertie d'après l'unité entre crochets."""
    chemin = os.path.join(path, nom)
    lignes, encodage = _lire_lignes(chemin)
    entete = next((i for i, l in enumerate(lignes[:200])
                   if _sans_accents(l).lstrip("\ufeff").startswith("date/time")), None)
    if entete is None:
        raise ValueError(f"En-tête 'Date/time' introuvable dans {nom}")

    df = pd.read_csv(chemin, sep=";", encoding=encodage, skiprows=entete, dtype=str, engine="python")
    df.columns = [c.strip() for c in df.columns]
    col = df.columns[0]
    brut = df[col].astype("string")
    fin = brut.map(lambda v: pd.notna(v) and "end of data" in _sans_accents(v)).fillna(False)
    df = df.loc[~(fin | brut.isna() | (brut.str.strip() == ""))].copy()

    df["Date/time"] = _en_datetime(df[col]).dt.round(PAS)
    for c in df.columns:
        if c not in ("Date/time", col):
            df[c] = pd.to_numeric(df[c].astype("string").str.strip()
                                  .str.replace(",", ".", regex=False), errors="coerce")
    for c in list(df.columns):
        if "cond" in _sans_accents(c):
            u = re.search(r"\[([^\]]*)\]", c)
            df[c] = df[c] * (1000.0 if u and _sans_accents(u.group(1)).startswith("ms/cm") else 1.0)
            df = df.rename(columns={c: "Cond_(µS/cm)"})
            break
    return df.loc[df["Date/time"].notna()].sort_values("Date/time")


#: Libellés VuSitu (sans numéro de série, sans accent) vers les noms du projet.
NOMS_TROLL = {
    "conductivite specifique (us/cm)":        "Cond_Troll_(µS/cm)",
    "temperature (c)":                        "température_Troll_(°C)",
    "turbidite (ntu)":                        "Turbidity_Troll_(NTU)",
    "concentration rdo (mg/l)":               "O2_Troll_(mg/l)",
    "saturation rdo (%sat)":                  "O2 (%Sat)",
    "fluorescence de chlorophylle-a (rfu)":   "FluorescenceChloro_a_Troll_(RFU)",
    "concentration de chlorophylle-a (ug/l)": "ConcentrationChloro_a_(µg/l)",
}


def lire_VuSitu(nom, path=VUSITU_PATH):
    """Export VuSitu : guillemets retirés, numéro de série retiré par regex."""
    lignes, _ = _lire_lignes(os.path.join(path, nom))
    df = pd.read_csv(StringIO("\n".join(l.replace('"', "") for l in lignes)), sep=",")
    cle = lambda c: (_sans_accents(re.sub(r"\s*\(\d{4,}\)\s*$", "", str(c)).strip())
                     .replace("\u03bc", "u").replace("\u00b5", "u").replace("\u00b0", ""))
    col = next((c for c in df.columns if "date" in _sans_accents(c)), df.columns[0])
    df["DATE"] = _en_datetime(df[col]).dt.round(PAS)
    return df.drop(columns=[col]).rename(
        columns={c: NOMS_TROLL[cle(c)] for c in df.columns if cle(c) in NOMS_TROLL})


#: Voies de la centrale. level, C1, T1 = SA sonde CTD.
#: C2, T2, Turbi, O2, Chlorophyl = le TROLL rapatrié, donc un doublon.
NOMS_OTT = {"level": "Niveau_CTDOTT_(cm)",
            "c1": "Cond_CTDOTT_(µS/cm)",   "t1": "Temp_CTDOTT_(°C)",
            "c2": "Cond_TrollOTT_(µS/cm)", "t2": "Temp_TrollOTT_(°C)",
            "turbi": "Turbidity_TrollOTT_(NTU)", "o2": "O2_TrollOTT_(mg/l)",
            "chlorophyl": "FluorescenceChloro_a_TrollOTT_(RFU)"}


def lire_OTT(nom, path=OTT_PATH):
    """Centrale, déjà en UTC : -99999 = absence, horodatages en double départagés."""
    chemin = os.path.join(path, nom)
    _, encodage = _lire_lignes(chemin)
    df = pd.read_csv(chemin, sep=";", encoding=encodage, dtype=str, engine="python")
    df.columns = [c.strip() for c in df.columns]
    col = df.columns[0]
    df["DATE"] = _en_datetime(df[col]).dt.round(PAS)
    df = df.drop(columns=[col])
    for c in df.columns:
        if c != "DATE":
            df[c] = pd.to_numeric(df[c].astype("string").str.strip()
                                  .str.replace(",", ".", regex=False), errors="coerce")
            df.loc[df[c].isin([-99999, -9999, 9999]), c] = np.nan
    df = df.rename(columns={c: NOMS_OTT[c.lower()] for c in df.columns if c.lower() in NOMS_OTT})
    return df.dropna(subset=["DATE"]).groupby("DATE", as_index=False).median(numeric_only=True)


def en_utc(df, nom, metadata, col_date="Date/time", col_utc="UTC Fichier"):
    """Ramène les horodatages en UTC. Correspondance EXACTE sur le nom du
    fichier : sinon la campagne est ignoree et le message la nomme."""
    ligne = metadata.loc[metadata["Nom fichier"] == nom, col_utc]
    if ligne.empty:
        raise ValueError(f"'{nom}' absent de la colonne 'Nom fichier'. "
                         f"À corriger dans la table UTC.")
    v = ligne.values[0]
    m = re.search(r"([+-]?\d+(?:[.,]\d+)?)", str(v))
    decalage = float(v) if isinstance(v, (int, float, np.number)) and pd.notna(v) else (
        float(m.group(1).replace(",", ".")) if m else 0.0)
    df = df.copy()
    df[col_date] = df[col_date] - pd.Timedelta(hours=decalage)
    return df, decalage


#: Gamme physique par mot-clé de nom de colonne. Pas de seuil sur le NIVEAU :
#: il n'a pas d'origine absolue tant qu'il n'est pas calé.
GAMMES = {"cond": (30, 5000), "temp": (-2, 30), "turbid": (0, 4000),
          "o2": (0, 25), "chloro": (0, 500)}


def appliquer_gammes(df):
    """Met à NaN ce qui est physiquement impossible, voie par voie."""
    for col in df.columns:
        if not pd.api.types.is_numeric_dtype(df[col]):
            continue
        for cle, (mini, maxi) in GAMMES.items():
            if cle in _sans_accents(col):
                hors = (df[col] < mini) | (df[col] > maxi)
                if hors.any():
                    print(f"  {col:36s} {int(hors.sum()):6d} hors [{mini}, {maxi}]")
                    df.loc[hors, col] = np.nan
                break
    return df

### Fonctions de correction

`decaler` porte le choix du sens : `aval` pour une marche réelle (capteur déplacé),
`amont` pour ramener l'historique sur la référence actuelle, `tout` pour un calage
global. On met à **NaN**, jamais de ligne supprimée.

In [ ]:
def graphe(traces, titre="", ylab="", points=None, col_point=None):
    """`traces` = liste de (série, nom, couleur). Scattergl : une chronique
    horaire pluriannuelle s'affiche sans saturer le navigateur."""
    fig = go.Figure()
    for serie, nom, couleur in traces:
        fig.add_trace(go.Scattergl(x=serie.index, y=serie, mode="lines", name=nom,
                                   line=dict(color=couleur, width=1.3)))
    if points is not None and col_point in points.columns:
        corr = points.get("Correction", pd.Series("Non", index=points.index))
        fig.add_trace(go.Scattergl(
            x=points["Datetime"], y=points[col_point], mode="markers", name="points de contrôle",
            marker=dict(symbol="x", size=10,
                        color=["red" if str(v).strip() == "Oui" else "royalblue" for v in corr])))
    fig.update_layout(title=titre, xaxis_title="Date", yaxis_title=ylab,
                      template="plotly_white", hovermode="x unified")
    fig.show()          # pas de `return fig` : sinon Jupyter réaffiche la figure


def decaler(serie, date, valeur, sens="tout"):
    """Ajoute `valeur` a toute la serie, a l'aval de `date` (incluse) ou a
    l'amont (strictement avant)."""
    if sens == "tout":
        return serie + valeur
    date = pd.to_datetime(date)
    m = np.asarray(serie.index >= date if sens == "aval" else serie.index < date)
    return serie.where(~m, serie + valeur)


def ecarter(serie, periodes):
    """Passe a NaN les periodes (debut, fin, motif) et le dit."""
    for debut, fin, motif in periodes:
        debut, fin = sorted([pd.to_datetime(debut), pd.to_datetime(fin)])
        m = np.asarray((serie.index >= debut) & (serie.index <= fin))
        print(f"  {debut:%d/%m/%Y %H:%M} - {fin:%d/%m/%Y %H:%M} : "
              f"{int((m & serie.notna().to_numpy()).sum())} pas écartés ({motif})")
        serie = serie.mask(m)
    return serie


def caler(serie, points, col_valeur, tolerance_h=1):
    """Recale la serie sur les mesures ponctuelles marquees Oui, en cascade
    vers l'aval. Chaque point est trace, applique ou non."""
    for _, l in points.dropna(subset=["Datetime"]).sort_values("Datetime").iterrows():
        date, cible = l["Datetime"], l.get(col_valeur)
        if pd.isna(cible) or str(l.get("Correction", "Non")).strip() != "Oui":
            continue
        mesures = serie.dropna()
        i = mesures.index[np.abs((mesures.index - date).to_numpy()).argmin()] if len(mesures) else None
        if i is None or abs((i - date).total_seconds()) > tolerance_h * 3600:
            print(f"  {date:%d/%m/%Y %H:%M} : ignoré, pas de mesure à moins de {tolerance_h} h")
            continue
        d = float(cible) - float(serie.loc[i])
        print(f"  {date:%d/%m/%Y %H:%M} : {float(serie.loc[i]):.1f} vers {float(cible):.1f}, "
              f"décalage {d:+.2f} appliqué vers l'aval")
        serie = decaler(serie, date, d, "aval")
    return serie


def fusionner(voies, periodes):
    """Chronique d'une grandeur. `voies` = {sonde: serie}, deja ramenees sur
    la meme reference. Sur chaque periode (debut, fin, sonde) la sonde
    indiquee est prioritaire ; les autres comblent ses trous, dans l'ordre du
    dictionnaire. Retourne la serie et la sonde retenue pas par pas."""
    index = next(iter(voies.values())).index
    valeur = pd.Series(np.nan, index=index)
    source = pd.Series(pd.NA, index=index, dtype="object")
    for debut, fin, prioritaire in periodes:
        p = np.asarray((index >= pd.to_datetime(debut)) & (index <= pd.to_datetime(fin)))
        for sonde in [prioritaire] + [s for s in voies if s != prioritaire]:
            trou = p & valeur.isna().to_numpy() & voies[sonde].notna().to_numpy()
            valeur[trou], source[trou] = voies[sonde][trou], sonde
    return valeur, source

## 4. CTD : lecture, UTC et compensation barométrique

`1 hPa = 1.019716 cmH2O`. Mettre `HPA_EN_CMH2O = 1.0` redonne l'ancienne formule.

In [ ]:
HPA_EN_CMH2O = 1.019716

metadata = pd.read_excel(UTC_CTD_PATH)
baro = pd.read_excel(BARO_PATH)[["DATE", BARO_COL]]
baro["DATE"] = pd.to_datetime(baro["DATE"], errors="coerce")
baro = baro.dropna(subset=["DATE"]).drop_duplicates("DATE")

morceaux = []
for nom in fichiers(CTD_PATH, PREFIXE_CTD):
    try:
        CTD, decalage = en_utc(lire_CTD(nom), nom, metadata)
    except Exception as e:
        print(f"  IGNORÉ  {nom} : {e}")
        continue
    m = pd.merge(CTD, baro, left_on="Date/time", right_on="DATE", how="left")
    m["Niveau_(cm)"] = m["Pression[cmH2O]"] - m[BARO_COL] * HPA_EN_CMH2O
    m = m.rename(columns={"Température[°C]": "Temp_(°C)"})
    morceaux.append(m[["Date/time", "Niveau_(cm)", "Cond_(µS/cm)", "Temp_(°C)"]])
    print(f"  {nom:45s} UTC+{decalage:g} vers UTC   ({len(CTD)} lignes)")

merge_ctd_df = pd.concat(morceaux, ignore_index=True).sort_values("Date/time", kind="stable")
merge_ctd_df["DATE"] = merge_ctd_df["Date/time"]
print(f"\n{len(morceaux)} campagne(s), {len(merge_ctd_df)} enregistrements en UTC.")

## 5. Raccordement à l'ancienne chronique

L'ancien fichier consolidé et les campagnes sont la **même sonde CTD**, séparées par
un trou d'exploitation. Le raccord se fait sur la jonction, comme la V2. Figer les
valeurs dans `DECALAGES_RACCORD` dès qu'elles conviennent.

In [ ]:
DECALAGES_RACCORD = {"Niveau": None, "Conductivité": None}    # None = recalculé

olddata_df = pd.read_excel(OLDDATA_PATH)
olddata_df["DATE"] = pd.to_datetime(olddata_df["DATE"], errors="coerce")
olddata_df = olddata_df.dropna(subset=["DATE"]).sort_values("DATE")

ancien, nouveau = olddata_df.set_index("DATE"), merge_ctd_df.set_index("DATE")
for nom, col_a, col_n, unite in [("Niveau", "Niveau_(cm)", "Niveau_(cm)", "cm"),
                                 ("Conductivité", "Cond_CTD_(µS/cm)", "Cond_(µS/cm)", "µS/cm")]:
    a, n = ancien[col_a].dropna(), nouveau[col_n].dropna()
    d = float(a.iloc[-1] - n.iloc[0]) if len(a) and len(n) else 0.0
    if DECALAGES_RACCORD.get(nom) is not None:
        print(f"{nom:13s} : {float(DECALAGES_RACCORD[nom]):+9.2f} {unite} figé "
              f"(jonction : {d:+.2f})")
        d = float(DECALAGES_RACCORD[nom])
    else:
        print(f"{nom:13s} : {d:+9.2f} {unite} (jonction {a.index[-1]:%d/%m/%Y %H:%M} "
              f"/ {n.index[0]:%d/%m/%Y %H:%M})")
    merge_ctd_df[col_n] = merge_ctd_df[col_n] + d

jonction = olddata_df["DATE"].max()
fig, ax = plt.subplots(figsize=(11, 3))
ax.plot(olddata_df["DATE"], olddata_df["Niveau_(cm)"], color="green", label="ancien")
ax.plot(merge_ctd_df["DATE"], merge_ctd_df["Niveau_(cm)"], color="blue", label="nouveau raccordé")
ax.set_xlim(jonction - pd.Timedelta(days=7), jonction + pd.Timedelta(days=7))
ax.set_ylabel("Niveau (cm)")
ax.legend()
plt.show()

## 6. TROLL : exports VuSitu

In [ ]:
metadata_troll = pd.read_excel(UTC_TROLL_PATH, sheet_name=0)

morceaux = []
for nom in fichiers(VUSITU_PATH, "VuSitu"):
    try:
        df_v, decalage = en_utc(lire_VuSitu(nom), nom, metadata_troll, col_date="DATE")
    except Exception as e:
        print(f"  IGNORÉ  {nom} : {e}")
        continue
    morceaux.append(df_v)
    print(f"  {nom:45s} UTC+{decalage:g} vers UTC   ({len(df_v)} lignes)")

merge_troll_df = (pd.concat(morceaux, ignore_index=True).dropna(subset=["DATE"])
                  .sort_values("DATE", kind="stable"))
print(f"\n{len(morceaux)} fichier(s), {len(merge_troll_df)} enregistrements en UTC.")

## 7. Centrale OTT

In [ ]:
merge_ott_df = (pd.concat([lire_OTT(nom) for nom in fichiers(OTT_PATH, "")], ignore_index=True)
                .groupby("DATE", as_index=False).median(numeric_only=True).sort_values("DATE"))
print(f"{len(merge_ott_df)} pas de temps, du {merge_ott_df['DATE'].min():%d/%m/%Y} "
      f"au {merge_ott_df['DATE'].max():%d/%m/%Y}")

## 8. Assemblage : une colonne par sonde

Trois lignées empilées sur la grille horaire. Sur un horodatage en double, la
**première valeur gagne**, comme la V2. Les voies du TROLL rapatriées par la
centrale sont réunies avec les exports directs **sans recalage** : même capteur.
L'écart médian entre les deux chemins est affiché, il doit être quasi nul.

In [ ]:
COLONNES_CTD   = ["Niveau_CTD_(cm)", "Cond_CTD_(µS/cm)", "Temp _CTD(°C)"]
COLONNES_TROLL = ["Cond_Troll_(µS/cm)", "température_Troll_(°C)", "Turbidity_Troll_(NTU)",
                  "O2_Troll_(mg/l)", "O2 (%Sat)", "FluorescenceChloro_a_Troll_(RFU)",
                  "ConcentrationChloro_a_(µg/l)"]
#: (voie directe, même voie rapatriée par la centrale)
DOUBLONS = [("Cond_Troll_(µS/cm)", "Cond_TrollOTT_(µS/cm)"),
            ("température_Troll_(°C)", "Temp_TrollOTT_(°C)"),
            ("Turbidity_Troll_(NTU)", "Turbidity_TrollOTT_(NTU)"),
            ("O2_Troll_(mg/l)", "O2_TrollOTT_(mg/l)"),
            ("FluorescenceChloro_a_Troll_(RFU)", "FluorescenceChloro_a_TrollOTT_(RFU)")]


def empiler(morceaux, colonnes):
    pile = pd.concat(morceaux, ignore_index=True).dropna(subset=["DATE"])
    pile = pile[["DATE"] + [c for c in colonnes if c in pile.columns]]
    return (pile.sort_values("DATE", kind="stable")
            .drop_duplicates("DATE", keep="first").set_index("DATE"))


piles = [
    empiler([olddata_df.rename(columns={"Niveau_(cm)": "Niveau_CTD_(cm)"}),
             merge_ctd_df.rename(columns={"Niveau_(cm)": "Niveau_CTD_(cm)",
                                          "Cond_(µS/cm)": "Cond_CTD_(µS/cm)",
                                          "Temp_(°C)": "Temp _CTD(°C)"})], COLONNES_CTD),
    empiler([olddata_df, merge_troll_df], COLONNES_TROLL),
    empiler([merge_ott_df], list(NOMS_OTT.values())),
]

grille = pd.date_range(min(p.index.min() for p in piles),
                       max(p.index.max() for p in piles), freq=PAS, name="DATE")
full_data = pd.DataFrame(index=grille)
for pile in piles:
    for col in pile.columns:
        full_data[col] = pile[col].reindex(grille)
print(f"{len(full_data)} pas horaires, du {grille.min():%d/%m/%Y} au {grille.max():%d/%m/%Y}")

print("Hors gamme physique :")
full_data = appliquer_gammes(full_data)

print("Voies TROLL rapatriées par la centrale :")
for direct, relais in DOUBLONS:
    commun = full_data[direct].notna() & full_data[relais].notna()
    trou = (full_data[direct].isna() & full_data[relais].notna()).to_numpy()
    ecart = (full_data[direct][commun] - full_data[relais][commun]).median()
    full_data[direct] = full_data[direct].where(~trou, full_data[relais])
    print(f"  {direct:34s} {int(trou.sum()):6d} pas repris  "
          f"(écart médian {ecart:+.2f} sur {int(commun.sum())} pas communs)")

#: Les sondes de chaque grandeur : {grandeur: {sonde: colonne}}.
SONDES = {
    "Niveau_(cm)":        {"OTT": "Niveau_CTDOTT_(cm)", "CTD": "Niveau_CTD_(cm)"},
    "Conductivité":       {"OTT": "Cond_CTDOTT_(µS/cm)", "TROLL": "Cond_Troll_(µS/cm)",
                           "CTD": "Cond_CTD_(µS/cm)"},
    "Température":        {"OTT": "Temp_CTDOTT_(°C)", "TROLL": "température_Troll_(°C)",
                           "CTD": "Temp _CTD(°C)"},
    "Turbidité_(NTU)":    {"TROLL": "Turbidity_Troll_(NTU)"},
    "O2_(mg/l)":          {"TROLL": "O2_Troll_(mg/l)"},
    "Chlorophylle_(RFU)": {"TROLL": "FluorescenceChloro_a_Troll_(RFU)"},
}
PARAMETRES = list(SONDES)

#: Voies écartées au jugement, AVANT la fusion : (début, fin, colonne, motif).
#: Une sonde qui dérive est écartée, une autre prend le relais.
VOIES_ECARTEES = [
    ("2023-09-05 13:00", "2023-10-21 22:00", "Temp _CTD(°C)", "dérive sonde CTD"),
    ("2021-04-09 14:00", "2021-06-03 12:00", "O2_Troll_(mg/l)", "capteur RDO défaillant"),
]
print("Voies écartées au jugement :")
for debut, fin, col, motif in VOIES_ECARTEES:
    full_data[col] = ecarter(full_data[col], [(debut, fin, f"{col} : {motif}")])

# Instantané des voies brutes. Les cellules de correction en repartent, jamais
# de la colonne qu'elles écrivent : relancer une cellule cumulerait les décalages.
BRUT = full_data.copy()
full_data.to_excel(SORTIE_CONSOLIDE)
print(f"\nDétail capteur par capteur : {SORTIE_CONSOLIDE}")

## 9. Proposition de périodes (à lancer une fois)

Écrit le code à **copier-coller** dans les cellules suivantes : découpage sur la
disponibilité réelle des sondes, et décalage médian de chaque sonde par rapport à la
sonde de référence. Ensuite ces listes vivent en dur dans le notebook : changer
l'ordre ne fait plus bouger les corrections déjà calées.

In [ ]:
ORDRE_PROPOSE = ["OTT", "TROLL", "CTD"]     # préférence pour la proposition
DUREE_MINI = pd.Timedelta("30D")            # blocs plus courts fondus dans le précédent
SUFFIXE = {"Niveau_(cm)": "NIVEAU", "Conductivité": "COND", "Température": "TEMPERATURE",
           "Turbidité_(NTU)": "TURBIDITE", "O2_(mg/l)": "O2", "Chlorophylle_(RFU)": "CHLORO"}

for grandeur, sondes in SONDES.items():
    ordre = [s for s in ORDRE_PROPOSE if s in sondes] + \
            [s for s in sondes if s not in ORDRE_PROPOSE]
    choix = pd.Series(pd.NA, index=full_data.index, dtype="object")
    for sonde in ordre:
        choix = choix.where(choix.notna() | full_data[sondes[sonde]].isna(), sonde)
    choix = choix.dropna()
    if choix.empty:
        continue

    blocs = []
    for _, g in choix.groupby((choix != choix.shift()).cumsum()):
        if blocs and (g.iloc[0] == blocs[-1][2] or g.index[-1] - g.index[0] < DUREE_MINI):
            blocs[-1][1] = g.index[-1]      # même sonde, ou bloc trop court : fondu
        else:
            blocs.append([g.index[0], g.index[-1], g.iloc[0]])
    blocs[-1][1] = pd.Timestamp("2100-01-01")   # la dernière période reste ouverte

    reference = blocs[-1][2]                    # la sonde en service aujourd'hui
    nom = SUFFIXE[grandeur]
    print(f"PERIODES_{nom} = [")
    for debut, fin, sonde in blocs:
        print(f'    ("{debut:%Y-%m-%d %H:%M}", "{fin:%Y-%m-%d %H:%M}", "{sonde}"),')
    print("]")
    ecarts = {}
    for sonde in sondes:
        if sonde == reference:
            continue
        a, b = full_data[sondes[reference]], full_data[sondes[sonde]]
        commun = a.notna() & b.notna()
        ecarts[sonde] = round(float((a[commun] - b[commun]).median()), 2) if commun.any() else 0.0
    print(f"DECALAGES_{nom} = {ecarts}    # ramène chaque sonde sur {reference}\n")

## 10. Niveau

`PERIODES_NIVEAU` dit quelle sonde est prioritaire sur quelle période. Elle est en
dur : changer l'ordre ne déplace pas le zéro, donc les corrections calées plus bas
restent valables.

Le **calage de la CTD** est figé lui aussi. L'échelle a été déplacée en juin 2024,
mais déplacer l'échelle ne déplace pas le capteur : la mesure est continue, il n'y a
pas de marche à corriger à cette date, seules les lectures au carnet changent de
référence. Le calage est donc ancré sur une lecture sûre et appliqué **au passé**
de celle-ci.

In [ ]:
# (date d'ancrage, décalage en cm, sens). Le 06/12/2024 16:00 UTC la centrale
# lisait 107 cm à l'échelle et la CTD 76.86 cm de moins. "amont" = appliqué
# avant cette date ; après, c'est la centrale qui fournit le niveau.
# La cellule 9 propose la valeur courante sous le nom DECALAGES_NIVEAU.
CALAGE_CTD = ("2024-12-06 16:00", 76.86, "amont")

PERIODES_NIVEAU = [
    ("2019-01-01 00:00", "2024-09-30 23:00", "CTD"),
    ("2024-10-01 00:00", "2100-01-01 00:00", "OTT"),
]
PERIODES_ECARTEES_NIVEAU = [
    ("2019-10-14 17:00", "2020-02-25 17:00", "sonde déplacée"),
]

points_niveau = pd.read_excel(PUNCTUAL_NIVEAU)
points_niveau["Datetime"] = pd.to_datetime(points_niveau["Jour"], dayfirst=True, errors="coerce")

date_ctd, valeur_ctd, sens_ctd = CALAGE_CTD
voies = {"OTT": BRUT["Niveau_CTDOTT_(cm)"],
         "CTD": decaler(BRUT["Niveau_CTD_(cm)"], date_ctd, valeur_ctd, sens_ctd)}
avant, source = fusionner(voies, PERIODES_NIVEAU)

print("Périodes écartées :")
niveau = ecarter(avant, PERIODES_ECARTEES_NIVEAU)
print("Points de contrôle :")
niveau = caler(niveau, points_niveau, "Hauteur (cm)")
full_data["Niveau_(cm)"], full_data["Niveau_(cm)_source"] = niveau, source
print("Pas de temps par sonde :", source.value_counts().to_dict())

graphe([(avant, "avant correction", "lightgrey"), (niveau, "après correction", "black")],
       titre="Niveau", ylab="Niveau (cm)", points=points_niveau, col_point="Hauteur (cm)")

## 11. Conductivité

`DECALAGES_COND` ramène chaque sonde sur la sonde de référence (la dernière en
service) : c'est ce qui permet de changer les périodes sans créer de marche.

In [ ]:
DECALAGES_COND = {"TROLL": 0.0, "CTD": 0.0}    # µS/cm, ajouté à chaque sonde
PERIODES_COND = [
    ("2019-01-01 00:00", "2021-05-31 23:00", "CTD"),
    ("2021-06-01 00:00", "2024-09-30 23:00", "TROLL"),
    ("2024-10-01 00:00", "2100-01-01 00:00", "OTT"),
]
PERIODES_ECARTEES_COND = [
    # ("2021-01-01 00:00", "2021-01-31 00:00", "motif"),
]
FENETRE_IQR, K_IQR, FIN_IQR = "800h", 1.5, "2021-02-14 23:59"

points_cond = pd.read_excel(PUNCTUAL_CONDUCT)
points_cond["Datetime"] = pd.to_datetime(points_cond["Jour"], dayfirst=True, errors="coerce")

voies = {s: BRUT[c] + DECALAGES_COND.get(s, 0.0) for s, c in SONDES["Conductivité"].items()}
avant, source = fusionner(voies, PERIODES_COND)

print("Périodes écartées :")
cond = ecarter(avant, PERIODES_ECARTEES_COND)
print("Points de contrôle :")
cond = caler(cond, points_cond, "Conductivité")

r = cond.rolling(FENETRE_IQR, center=True, min_periods=8)
q1, q3 = r.quantile(0.25), r.quantile(0.75)
hors = (np.asarray(((cond < q1 - K_IQR * (q3 - q1)) | (cond > q3 + K_IQR * (q3 - q1))).fillna(False))
        & np.asarray(full_data.index <= pd.to_datetime(FIN_IQR)))
cond = cond.mask(hors)
print(f"Filtre IQR ({FENETRE_IQR}, k={K_IQR}, jusqu'au {FIN_IQR}) : "
      f"{int(hors.sum())} valeurs écartées")

full_data["Conductivité"], full_data["Conductivité_source"] = cond, source
full_data["Conductivité_Moyenne_Mobile"] = cond.rolling("6h", center=True).mean()
print("Pas de temps par sonde :", source.value_counts().to_dict())

graphe([(avant, "avant correction", "lightgrey"), (cond, "après correction", "black")],
       titre="Conductivité", ylab="Conductivité (µS/cm)",
       points=points_cond, col_point="Conductivité")

## 12. Température et autres paramètres

Pas de correction ici, seulement la fusion. Les voies défaillantes ont déjà été
écartées en cellule 8.

In [ ]:
DECALAGES = {"Température": {"TROLL": 0.0, "CTD": 0.0}}
PERIODES = {
    "Température": [
        ("2019-01-01 00:00", "2021-05-31 23:00", "CTD"),
        ("2021-06-01 00:00", "2024-09-30 23:00", "TROLL"),
        ("2024-10-01 00:00", "2100-01-01 00:00", "OTT"),
    ],
    "Turbidité_(NTU)":    [("2019-01-01 00:00", "2100-01-01 00:00", "TROLL")],
    "O2_(mg/l)":          [("2019-01-01 00:00", "2100-01-01 00:00", "TROLL")],
    "Chlorophylle_(RFU)": [("2019-01-01 00:00", "2100-01-01 00:00", "TROLL")],
}

for grandeur, periodes in PERIODES.items():
    voies = {s: BRUT[c] + DECALAGES.get(grandeur, {}).get(s, 0.0)
             for s, c in SONDES[grandeur].items()}
    full_data[grandeur], full_data[f"{grandeur}_source"] = fusionner(voies, periodes)
    print(f"{grandeur:20s} {full_data[grandeur].notna().sum():6d} pas  "
          f"{full_data[f'{grandeur}_source'].value_counts().to_dict()}")

## 13. Cote NGF, interpolation et statuts

`FORMULE_NGF = "V2"` reprend la formule d'origine, dont le zéro implicite est à
**106.5396** m NGF et non 107.6158 ; `"zéro"` place le zéro à 107.6158. Écart de
1.076 m, à trancher avec le relevé topographique.

Les lacunes de moins de 12 h sont comblées. `Statut_<grandeur>` dit si la valeur est
mesurée, interpolée ou manquante. La cote se recalcule depuis le niveau interpolé.

In [ ]:
NIVEAU_NGF_CABOUY = 107.6158
FORMULE_NGF = "V2"
MAX_TROU_H = 12

cote_ngf = (lambda h: NIVEAU_NGF_CABOUY + h / 100) if FORMULE_NGF == "zéro" else \
           (lambda h: NIVEAU_NGF_CABOUY - (NIVEAU_NGF_CABOUY - h) / 100)

max_pas = int(pd.Timedelta(f"{MAX_TROU_H}h") / pd.Timedelta(PAS))
for col in PARAMETRES:
    origine = full_data[col]
    manquant = origine.isna().to_numpy()
    groupe = np.cumsum(np.r_[True, manquant[1:] != manquant[:-1]])
    tailles = pd.Series(groupe).groupby(groupe).transform("size").to_numpy()
    comble = origine.interpolate(method="time", limit_direction="both")
    comble = comble.mask(manquant & (tailles > max_pas))
    mesures = np.flatnonzero(~manquant)          # pas d'extrapolation hors plage mesurée
    if mesures.size:
        comble.iloc[:mesures[0]] = origine.iloc[:mesures[0]]
        comble.iloc[mesures[-1] + 1:] = origine.iloc[mesures[-1] + 1:]
    full_data[col] = comble
    full_data[f"Statut_{col}"] = np.where(
        ~manquant, "Mesurée", np.where(comble.notna().to_numpy(), "Interpolée", "Manquante"))

full_data["Niveau_(mNGF)"] = cote_ngf(full_data["Niveau_(cm)"])
full_data["Statut_Niveau_(mNGF)"] = full_data["Statut_Niveau_(cm)"]
print(f"Formule NGF '{FORMULE_NGF}' : zéro d'échelle à {cote_ngf(0):.4f} m NGF")
display(pd.DataFrame({c: full_data[f"Statut_{c}"].value_counts()
                      for c in PARAMETRES}).fillna(0).astype(int).T)

## 14. Sauvegarde et graphe de synthèse

Un paramètre par grandeur, avec son statut et la sonde qui l'a fourni. Le détail
capteur par capteur reste dans le fichier écrit en cellule 8.

In [ ]:
finaux = PARAMETRES + ["Niveau_(mNGF)"]
colonnes = [c for p in finaux for c in (p, f"Statut_{p}", f"{p}_source") if c in full_data]
full_data[colonnes].to_excel(SORTIE_FINALE)
print(f"{SORTIE_FINALE} : {len(full_data)} pas x {len(colonnes)} colonnes")

fig, axes = plt.subplots(3, 1, figsize=(15, 10), sharex=True, gridspec_kw={"hspace": 0.05})
axes[0].plot(full_data.index, full_data["Niveau_(cm)"].rolling(12, center=True).mean(),
             color="lightseagreen")
axes[0].set_ylabel("Niveau (cm)", color="lightseagreen")
if os.path.exists(PLUIE_PATH):
    pluie = pd.read_csv(PLUIE_PATH)
    ax = axes[0].twinx()
    ax.bar(pd.to_datetime(pluie["Date"], errors="coerce"), pluie["Precipitation (mm)"],
           width=0.8, color="royalblue")
    ax.invert_yaxis()
    ax.set_ylabel("Précipitations (mm)", color="royalblue")

axes[1].plot(full_data.index, full_data["Conductivité_Moyenne_Mobile"], color="black")
axes[1].set_ylabel("Conductivité (µS/cm)")
ax = axes[1].twinx()
ax.plot(full_data.index, full_data["Température"].rolling(12, center=True).mean(), color="crimson")
ax.set_ylabel("Température (°C)", color="crimson")

axes[2].plot(full_data.index, full_data["Turbidité_(NTU)"].rolling(24, center=True).mean(),
             color="darkorange")
axes[2].set_ylabel("Turbidité (NTU)", color="darkorange")
axes[2].set_ylim(0, 100)
ax = axes[2].twinx()
ax.plot(full_data.index, full_data["O2_(mg/l)"].rolling(24, center=True).mean(),
        color="darkmagenta")
ax.set_ylabel("Oxygène (mg/L)", color="darkmagenta")

plt.savefig(SORTIE_SVG, format="svg")
plt.show()